In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# load dataset
tips = sns.load_dataset("tips")

# tips.head()
# # set the figure size
# plt.figure(figsize=(14, 14))

# # from raw value to percentage
total = tips.groupby('day')['total_bill'].sum().reset_index()
smoker = tips[tips.smoker=='Yes'].groupby('day')['total_bill'].sum().reset_index()
smoker
smoker['total_bill'] = [i / j * 100 for i,j in zip(smoker['total_bill'], total['total_bill'])]
smoker
total['total_bill'] = [i / j * 100 for i,j in zip(total['total_bill'], total['total_bill'])]
total

# # bar chart 1 -> top bars (group of 'smoker=No')
# bar1 = sns.barplot(x="day",  y="total_bill", data=total, color='darkblue')

# # bar chart 2 -> bottom bars (group of 'smoker=Yes')
# bar2 = sns.barplot(x="day", y="total_bill", data=smoker, color='lightblue')

# # add legend
# top_bar = mpatches.Patch(color='darkblue', label='smoker = No')
# bottom_bar = mpatches.Patch(color='lightblue', label='smoker = Yes')
# plt.legend(handles=[top_bar, bottom_bar])

# # show the graph
# plt.show()

# Final Project Template

For the final project for this module, you are asked to use ETL together with the skills you have learned about Python and MySQL in the previous modules to understand spending patterns.

This module's project is divided into two main parts: Extract-Transform-Load (ETL) and Analysis and Visualization.

Your challenge in this project is to implement the steps suggested by Dr. Sanchez in his videos throughout the module and prove that you have a a clear understanding of each of them by being able to describe and justify them. You will also be tested on your ability to conduct your own analysis to understand spending patterns.

Before you fill out the project outline template below, make sure you:

- Read through the template completely to understand the instructions for the structure of the project.
- Have a clear understanding of what to do to create a model that will return the results you want to find.
- Use Markdown to edit the template.
- Include any screenshots of your code (both Python and MySQL) and of your program windows (Excel, Terminal, VS Code, MySQL Workbench) to demonstrate your steps.

<div class="alert alert-block alert-success">
The purpose of this Jupyter Notebook is to give you a structure to follow when you are solving your problem and developing your model with Python. Make sure you follow it carefully. You can add more subsections if needed, but remember to fill out every section provided in the template.
</div>

<div class="alert alert-block alert-danger">
Delete all cells above, including this one, before submitting your final Notebook.
</div>

# Title

**Your_Name**

Add the title of your project and delete the default one.

# Index

- [Abstract](#Abstract)
- [1. Introduction](#1.-Introduction)
- [2. Extract-Transform-Load](#2.-Extract-Transform-Load)
    - [2.1 The ETL Process](#2.1-The-ETL-Process)
    - [2.2 Data Exploration](#2.2-Data-Exploration)
    - [2.3 Data Preparation](#2.3-Data-Preparation)
    - [2.4 Read the Data Using Python](#2.4-Reading-the-Data-Using-Python)
         - [2.4.1 Reading Sample Data](#2.4.1-Reading-Sample-Data)
         - [2.4.2 Reading the MRST Data](#2.4.2-Reading-the-MRST-Data)
    - [2.5 Writing an Installation Script](#2.5-Writing-an-Installation-Script)
- [3. Analysis and Visualization](#3.-Project-Description)
    - [3.1 Running Queries in MySQL Workbech](#3.1-Running-Queries-in-MySQL-Workbech)
    - [3.2 Running Queries From Python](#3.2-Running-Queries-From-Python)
    - [3.3 Explore Trends](#3.3-Explore-Trends)
    - [3.4 Explore Percentage Change](#3.4-Explore-Percentage-Change)
    - [3.5 Explore Rolling Time Windows](#3.5-Explore-Rolling-Time-Windows)
- [Conclusion](#Conclusion)
- [References](#References)

[Back to top](#Index)


##  Abstract

This is a brief description (150 words or less) of your analysis and the results of your model. Complete this portion of the template after you are done working on your project.

Included all four aspects of the final project and clearly and succinctly summarized the: 
- Purpose of the project 
- Models or methods used in the project 
- Major findings from the analysis 
- Interpretations and conclusions

[Back to top](#Index)


## 1. Introduction

Introduce your project using 300 words or less. Describe all the processes you followed to create your ETL, Analysis, and Visualization project. Start by summarizing the steps that you intend to perform and then elaborate on this section after you have completed your project.

Introduced the project and provided a detailed description of the required sections: 
- An overview of the problem-solving framework 
- Workflow model 
- Processes used to solve the identified data engineering problem

[Back to top](#Index)

## 2. Extract-Transform-Load

[Back to top](#Index)

### 2.1 The ETL Process

Key Steps for processing MRTS dataset:
- read each tab in the .xls file into individual csv files.
- clean data.  See section "2.3 Data Preparation" for details.
- create installation script for MySQL
- connect to MySQL
- run installation script



Described the key steps to perform ETL on a general dataset in detail by including personal insights.

Describe, using your own words, the key steps to perform ETL on the provided MRTS dataset.

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import os
import mysql.connector

In [ ]:
pd.set_option('display.max_rows',200)

[Back to top](#Index)

### 2.2 Data Exploration

#### !!!Described the dataset accurately and demonstrated that independent research was conducted to familiarize with the data.

Did a lot of exploration of the NAICS dataset, and trends presented on St. Louis Fed and the Bureau of Labor Statistics websites.  The NAICS dataset for MRTS is only concerned with NAICS sectors 44 and 45 (Retail Trade), and subsector 722 (Food Services and Drinking Places).

I went to the St Louis Fed website to get the seasonaly adjusted monthly CPI index so i could adjust the amounts into Feb 2021 dollars to show actual vs inflation adjusted trends.

I found it interesting, but not surprising, that there were choices made in 1992 with the first publication that have followed through to today.  For instance, including 4411 and 4412 on the same line, but not breaking them out on subsequent lines.  I assume they continued those choices all the way to today for consisitency, but i would be curious to know why those decisions were made.

[Back to top](#Index)

### 2.3 Data Preparation

#### !!!Described and justified precisely which modifications were made to the dataset and included one or more modifications not demonstrated in the video.

My goal was to put all of the csv files into one table with the grain of month_year and category.  
the columns would be: 
- NAICS_code_class: the top level description of each code length.
- Business Type: NAICS category name
- month_year: the month and year for the sample. 

- year
- month

- amount: dollar amount of sample
- adjusted_amount: this is CPI inflation (all minus food and energy) adjusted and seasonaly adjusted.

In that format, it will be easier to do some time series analysis.

[Back to top](#Index)

### 2.4 Read the Data Using Python

The Pandas library is great for exactly this sort of thing. Once your data is read into a dataframe, pandas can be used to do a lot, if not all of your data cleaning tasks.  I am not normally used to data cleaning with anything by SQL, so i have been impressed with the Pandas versitiliy.  Below you will see the code i used to clean each of the individual yearly csv files.

[Back to top](#Index)

### 2.4.1 Reading Sample Data

I have used the 2018 data in csv form.  No particular reason.  It was the first complete year that i started working with and just continued with it because it seemed like a good testbed.

I also used the CPI data described below to enhance the data.  I downloaded it from the site and created the monthly ratios using the formula:  
        **(feb 2021 index / each monthly index) *  amount**    
The first term gets us the ratio of month to feb 2021.  Then multiplying the amounts of each category in the MRTS sheets gets us the adjusted amounts.

I then used the code for the 2018 cleaning research to create the overall cleaning script in 2.4.2 below.  I have fully commented the code so you can follow along.

[Back to top](#Index)

### 2.4.2 Reading the MRTS Data

Step by Step description of the cleaning script below.  The code is fully commented also.
- break up xls file into individual yearly csv files
- for each yearly csv file:
    - load file
    - update column names
    - add surogate keys for first 7 lines
    - keep rows of unadjusted data and drop the rest
    - cast columns as numeric
    - disaggregate '4411,4412' row and add new 4412 row
    - drop rows '4411,4412' and '442,443' as they are unneeded
    - adjusted NAICS code for row '722513, 722514, 722515'
    - fix '(NA)' and '(S)' cells as NaN
    - reset index
    - add df to dict
    - use pd.melt() to unpivot the data into the grain:
        - NAICS_code
        - month_year
- concat the year dfs into one full_df
- add columns:
    - year
    - month
    - month_start (used this one to merge with inflation adjustment df)
- load inflation adjustment csv into df
- merge inflation adjustment df into full_df
- add inflation adjusted dollar amount column
- create final_df by selecting only needed columns from merged df
- write final_df to csv file


In [ ]:
# First, we need to loop through the tabs in the .xls sheet and save them as seperate csv files.
xls_file = 'MRTS/mrtssales92-present.xls'

xls = pd.ExcelFile(xls_file, engine='xlrd')

for sheet_name in xls.sheet_names:
    df = xls.parse(sheet_name)
    csv_file = f"MRTS/MRTS_year_files/MRTS_{sheet_name}.csv"
    df.to_csv(csv_file, index=False)

    print(f"Saved {sheet_name} to {csv_file}")

In [ ]:
# list of basic column names to be customized later
col_names=['NAICS_code', 'Business_type', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December', 'TOTAL', 'Feb. 2021(p)']

# list of column names that are edge cases that need to be removed
removed_col_names = ['CY CUM', 'PY CUM', 'Feb. 2021(p)_2016', 'Unnamed: 15']

file = 'MRTS_2018.csv'
year = file[5:9]
df_col_names = []

new_df = pd.read_csv(f'MRTS/MRTS_year_files/{file}', skiprows=[0,1,2,3,5] ,header=0)#, names=['NAICS_code', 'Business_type'] + df_col_names)
n_col = 0
for n in new_df.columns:
    if n not in removed_col_names:
        n_col+=1
for n in col_names[2:n_col]:
        df_col_names.append(f'{n}_{year}')
new_df = new_df.iloc[:,:n_col]
        
new_col_names = ['NAICS_code', 'Business_type'] + df_col_names
new_df.columns = new_col_names

# add "sector" keys to the 7 aggregate values at the top of the sheet.
# I am keeping them because it might be easier to do time series analysis using preaggregated than building the aggregations myself.
new_df.iloc[0:7,0] = range(1,8)

#removing the bottom adjusted data and all of the rest of the unneccesary rows.
new_df = new_df[:65]

# I need to reduce the last code in the series to only one NAICS code.
# There is no way to break out or determine which code to use.
# After doing some research, it makes sense to use 722513 as the stand-in for these codes.
new_df.iloc[-1,0] = 722513

# replacing '(S)' values with 0 for now so i can convert all of the columns to int data type.
# I will get rid of those cells when i put everything together.
new_df[new_df.iloc[:,2:] == '(S)'] = np.nan
new_df[new_df.iloc[:,2:] == '(NA)'] = np.nan

# converting columns to numeric
for col in new_df.columns[2:]:
    new_df[col] = pd.to_numeric(new_df[col])
    
# codes 4411 and 4412 share a line.  4411 is browken out individually, but not 4412.
# I decided to create a row that subtracts 4411 from the total to get 4412 by itself.
new_row = {}
new_row['NAICS_code']=4412
new_row['Business_type']='Other Motor Vehicle Dealers'
for col in new_df.columns[2:]:
    new_row[col] = new_df.iloc[8][col]-new_df.iloc[9][col]
new_df.loc[len(new_df)] = new_row

# now that i have the disggregated row for 4412, i can get rid of the preaggregated row.
# dropping NAICS_code=='4411,4412' because, now that we have split it out, we dont need this line.
new_df.drop(new_df[new_df.NAICS_code=='4411,4412'].index, inplace=True)

# dropping NAICS_code=='442,443' because both subcatagories have already been broken out below, so this line isnt needed.
new_df.drop(new_df[new_df.NAICS_code=='442,443'].index, inplace=True)

# reset index
new_df.reset_index(inplace=True)
new_df.drop(columns=['index'])


            

In [ ]:

# Define the path to your output text file
output_file_path = 'MRTS/MRTS_year_files/naics_data_insert_sql.txt'

# Generate the column names for the SQL statement
insert_cols = ",".join([str(i) for i in df.columns.tolist()])

# # Open the output text file in write mode
# with open(output_file_path, 'w') as file:
#     # Iterate through each row in the DataFrame
#     for i, row in df.iterrows():
#         # Create the SQL INSERT statement
#         sql = "INSERT INTO your_table_name (" + insert_cols + ") VALUES (" + "%s,"*(len(row)-1) + "%s);" % tuple(row)
#         # Write the SQL statement to the text file
#         file.write(sql + "\n")

In [ ]:
# Full cleaning script

# create dict to hold all dataframes
df_dict = {}

# list of basic column names to be customized later
col_names=['NAICS_code', 'Business_type', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December', 'TOTAL', 'Feb. 2021(p)']

# list of column names that are edge cases that need to be removed
removed_col_names = ['CY CUM', 'PY CUM', 'Feb. 2021(p)_2016', 'Unnamed: 15']

'''
main cleaning script:
for each yearly csv file:
    load file
    update column names
    add surogate keys for first 7 lines
    keep rows of unadjusted data and drop the rest
    cast columns as numeric
    disaggregate 4411,4412 row and add new 4412 row
    drop unneeded rows
    fix '(NA)' and '(S)' cells as NaN
    reset index
    add df to dict
'''
for file in os.listdir('MRTS/MRTS_year_files'):
    year = file[5:9]
    df_col_names = []

    new_df = pd.read_csv(f'MRTS/MRTS_year_files/{file}', skiprows=[0,1,2,3,5] ,header=0)#, names=['NAICS_code', 'Business_type'] + df_col_names)
    n_col = 0
    for n in new_df.columns:
        if n not in removed_col_names:
            n_col+=1
    for n in col_names[2:n_col]:
         df_col_names.append(f'{n}_{year}')
    new_df = new_df.iloc[:,:n_col]
         
    new_col_names = ['NAICS_code', 'Business_type'] + df_col_names
    new_df.columns = new_col_names

    # add "sector" keys to the 7 aggregate values at the top of the sheet.
    # I am keeping them because it might be easier to do time series analysis using preaggregated than building the aggregations myself.
    new_df.iloc[0:7,0] = range(1,8)

    #removing the bottom adjusted data and all of the rest of the unneccesary rows.
    new_df = new_df[:65]

    # I need to reduce the last code in the series to only one NAICS code.
    # There is no way to break out or determine which code to use.
    # After doing some research, it makes sense to use 722513 as the stand-in for these codes.
    new_df.iloc[-1,0] = 722513

    # replacing '(S)' values with 0 for now so i can convert all of the columns to int data type.
    # I will get rid of those cells when i put everything together.
    new_df[new_df.iloc[:,2:] == '(S)'] = np.nan
    new_df[new_df.iloc[:,2:] == '(NA)'] = np.nan

    # converting columns to numeric
    for col in new_df.columns[2:]:
        new_df[col] = pd.to_numeric(new_df[col])
        
    # codes 4411 and 4412 share a line.  4411 is browken out individually, but not 4412.
    # I decided to create a row that subtracts 4411 from the total to get 4412 by itself.
    new_row = {}
    new_row['NAICS_code']=4412
    new_row['Business_type']='Other Motor Vehicle Dealers'
    for col in new_df.columns[2:]:
        new_row[col] = new_df.iloc[8][col]-new_df.iloc[9][col]
    new_df.loc[len(new_df)] = new_row

    # now that i have the disggregated row for 4412, i can get rid of the preaggregated row.
    # dropping NAICS_code=='4411,4412' because, now that we have split it out, we dont need this line.
    new_df.drop(new_df[new_df.NAICS_code=='4411,4412'].index, inplace=True)

    # dropping NAICS_code=='442,443' because both subcatagories have already been broken out below, so this line isnt needed.
    new_df.drop(new_df[new_df.NAICS_code=='442,443'].index, inplace=True)

    # reset index
    new_df.reset_index(inplace=True)
    new_df.head(15)

    df_dict[f'df_{year}'] = new_df
    # print(df_dict[f'df_{year}'].columns)

# unpivoting eaech file into NAICS_code, month_year grain
# list for conating dfs
melted_list = []

for file in os.listdir('MRTS/MRTS_year_files'):
    # create list of value columns to unpivot
    year = file[5:9]
    a = df_dict[f'df_{year}'].columns[3:]

    # melt (unpivot) each file
    df_dict[f'df_{year}_melted'] = pd.melt(df_dict[f'df_{year}'],
                                          id_vars=['NAICS_code', 'Business_type'], 
                                          value_vars=a,
                                          var_name='month_year', 
                                          value_name='Raw_Amount')
    # building final df
    melted_list.append(df_dict[f'df_{year}_melted'])
    mrts_df = pd.concat(melted_list, ignore_index=True)
# adding month and year columns for easier analysis
mrts_df['year'] = mrts_df['month_year'].str[-4:]
mrts_df['month'] = mrts_df['month_year'].str[:-5]

# add month_start date column to join on cpi_df
month_dict = dict(zip(pd.date_range('1992-01-01', freq='ME', periods=12).strftime('%B'),
             pd.date_range('1992-01-01', freq='ME', periods=12).strftime('%m') + '-01'))
mrts_df['month_start'] = mrts_df['year'] + '-' + final_df['month'].map(month_dict)

In [ ]:
mrts_df.head()

In [ ]:
# read in CPI adjustment data csv
cpi_df = pd.read_csv('MRTS/CPI_Index_calc.csv')

In [ ]:
# merge inflation adjustment data and mrts data
full_df = pd.merge(mrts_df, cpi_df[['DATE', 'feb2021_multiple']], left_on='month_start', right_on='DATE')

# add inflation adjustment column.
full_df['2021_infl_seas_adjd'] = round(full_df['Raw_Amount'] * full_df['feb2021_multiple'])

# creating table in its final form
final_df = full_df[['NAICS_code', 'Business_type', 'month_year', 'year', 'month', 'month_start', 'Raw_Amount', '2021_infl_seas_adjd']]

In [ ]:
# writing new csv file for import to MySQL
csv_file = f"MRTS/MRTS_year_files/MRTS_Data.csv"
final_df.to_csv(csv_file, index=False)

[Back to top](#Index)

### 2.5 Writing an Installation Script

Describe how you wrote a Python installation script to read your dataset in MySQL WorkBench.

In [ ]:


cnx = mysql.connector.connect(
    user = 'root',
    password = '7890oiuy&*()OIUY',
    host = '127.0.0.1',
    #   database = 'sakila',
    auth_plugin = 'mysql_native_password'
)

cursor = cnx.cursor()
query = ("select 'this is open'")
cursor.execute(query)

# for row in cursor.fetchall():
#     print(row)

cursor.close()
cnx.close()


[Back to top](#Index)

## 3. Analysis and Visualization

For each of the sections below, make sure you include a description of the steps you followed. Whenever possible, include screenshots of your code or program windows to demonstrate your steps.

Here, describe the differences, advantages, and disadvantages of running *queries* against your dataset using the MySQL Workbench or a Python environment.

[Back to top](#Index)

### 3.1 Running Queries in MySQL Workbech

Describe which *queries* you ran against the MRTS dataset in MySQL Workbench to verify that everything worked as expected.

[Back to top](#Index)

### 3.2 Running Queries From Python

Describe how you tested the previous *queries* on the the MRTS dataset using a Python script and the Terminal window. 

[Back to top](#Index)

### 3.3 Explore Trends

Describe which *queries* you wrote the explore the differences in trends between various categories in your data.

In your submission make sure to answer the following:

- What is an economic trend and why is it considered an important measure to predict quantities, like spending patterns?
- What is the trend of the retail and food services categories? Can this data be displayed clearly or do you need to adjust some parameters to reduce extraneous details and be able to visualize a clean trend?
- When comparing businesses like bookstores, sporting goods stores, and hobbies, toys, and games stores, what is the highest trend of all of these options? Which one grew faster? Which one is higher? Is there a seasonal pattern? Were there any changes in 2020? Which is better, monthly or yearly? 

[Back to top](#Index)

### 3.4 Explore Percentage Change

Describe which *queries* you wrote to explore the differences in trends between various categories in your data.

In your submission make sure to answer the following:

- In economics, what is the percentage change and why is it considered an important measure to predict quantities like spending patterns?
- Consider the women's clothing and men's clothing businesses and their percentage change. How are these two businesses related? For each of the two businesses, what is the percentage of contribution to the whole and how does it change over time?

[Back to top](#Index)

### 3.5 Explore Rolling Time Windows


Describe which *queries* you wrote to explore the differences in trends between various categories in your data.

In your submission, make sure to answer the following:

- In economics, what is the rolling time window and why is it considered an important measure to predict quantities like spending patterns?
- Consider at least two businesses of your own from the MRTS data. Which *queries* did you write to analyze and produce graphs of rolling time windows for the chosen categories?

[Back to top](#Index)

## Conclusion

Describe your conclusions. Which one of the businesses considered seems like it's going to attract the least spending? Which business seems likely to attract the most spending? 





[Back to top](#Index
)
## References

Add all references you used to complete this project.

Use this format for articles:
- Author Last Name, Author First Name. ?Article Title.? Journal Title Volume #, no. Issue # (year): page range.

- Ex: Doe, John. ?Data Engineering.? Data Engineering Journal 18, no. 4 (2021): 12-18.

Use this format for websites:
- Author Last Name, Author First Name. ?Title of Web Page.? Name of Website. Publishing organization, publication or revision date if available. Access date if no other date is available. URL .

- Doe, John. ?Data Engineering.? Data Engineer Resource. Cengage, 2021. www.dataengineerresource.com .
